In [9]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
from dataset import FaceDataset,get_transforms
from model import get_efficientnet,get_xception
from utils import save_metrics,plot_confusion_matrix
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR="/kaggle/input/fdsdfdsdfg/faces_split/faces_split"
BATCH_SIZE=32
OUTPUT_DIR="outputs"
os.makedirs(OUTPUT_DIR,exist_ok=True)
transform=get_transforms()
test_ds=FaceDataset(os.path.join(DATA_DIR,"test"),transform)
if len(test_ds)==0:
    raise ValueError("❌ Test dataset is empty. Check faces_split/test structure.")
test_loader=DataLoader(test_ds, batch_size=BATCH_SIZE)
def evaluate_model(model, weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    y_true,y_pred=[],[]
    with torch.no_grad():
        for imgs,labels in test_loader:
            imgs=imgs.to(DEVICE)
            outputs=model(imgs).squeeze()
            preds=(torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
    if len(y_true)==0:
        raise ValueError("❌ No predictions made. Test loader empty.")
    metrics={
        "accuracy":accuracy_score(y_true,y_pred),
        "precision":precision_score(y_true,y_pred),
        "recall":recall_score(y_true,y_pred),
        "f1_score":f1_score(y_true,y_pred)
    }
    return metrics,y_true,y_pred
eff_model=get_efficientnet()
eff_metrics,eff_y_true,eff_y_pred=evaluate_model(eff_model,"/kaggle/input/fdsdfdsdfg/efficientnet.pth")
xcp_model=get_xception()
xcp_metrics,xcp_y_true,xcp_y_pred=evaluate_model(xcp_model, "/kaggle/input/fdsdfdsdfg/xception.pth")
final_metrics={
    "EfficientNet": eff_metrics,
    "XceptionNet": xcp_metrics
}
save_metrics(final_metrics,os.path.join(OUTPUT_DIR,"metrics.json"))
plot_confusion_matrix(eff_y_true,eff_y_pred,os.path.join(OUTPUT_DIR,"confusion_matrix.png"))
print("\n📊 FINAL COMPARISON")
print(f"{'Model':<15}{'Acc':<10}{'Prec':<10}{'Recall':<10}{'F1':<10}")
print(f"{'EfficientNet':<15}"
      f"{eff_metrics['accuracy']:<10.4f}"
      f"{eff_metrics['precision']:<10.4f}"
      f"{eff_metrics['recall']:<10.4f}"
      f"{eff_metrics['f1_score']:<10.4f}")
print(f"{'XceptionNet':<15}"
      f"{xcp_metrics['accuracy']:<10.4f}"
      f"{xcp_metrics['precision']:<10.4f}"
      f"{xcp_metrics['recall']:<10.4f}"
      f"{xcp_metrics['f1_score']:<10.4f}")
print("\n✅ Evaluation complete!")
print("📁 Saved: outputs/metrics.json")
print("📁 Saved: outputs/confusion_matrix.png")


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(



📊 FINAL COMPARISON
Model          Acc       Prec      Recall    F1        
EfficientNet   0.9414    0.9594    0.9209    0.9398    
XceptionNet    0.8160    0.7928    0.8518    0.8212    

✅ Evaluation complete!
📁 Saved: outputs/metrics.json
📁 Saved: outputs/confusion_matrix.png
